## Model Training
Trains a BiLSTM + temporal-attention classifier for 6 cricket shot classes with label smoothing and mixup augmentation.

Note: Backward Defence is excluded upstream in notebook-1.ipynb. The labels in y.npy are already 0-indexed {0…5} = [Cover Drive, Cut, Flick, Forward Defence, Pull, Sweep].

In [1]:
import numpy as np
import torch
import torch.nn as nn
import json
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from torch.utils.data import TensorDataset, DataLoader

In [2]:
X = np.load(r'D:\CA_POC\new_processed\X_features.npy')   # (N, 100, 214)
y = np.load(r'D:\CA_POC\new_processed\y.npy')

# Backward Defence is already excluded in notebook-1.ipynb.
# y contains exactly {0,1,2,3,4,5} mapped to the 6 classes below.
print('Shape:', X.shape, y.shape)
print('Class distribution:', Counter(y.tolist()))

assert X.shape[2] == 214, f"Expected 214 features, got {X.shape[2]}"

LABEL_NAMES = [
    'Cover Drive',       # 0
    'Cut',               # 1
    'Flick',             # 2
    'Forward Defence',   # 3
    'Pull',              # 4
    'Sweep',             # 5
]
NUM_CLASSES = len(LABEL_NAMES)
assert NUM_CLASSES == len(np.unique(y)), (
    f"Label count mismatch: expected {NUM_CLASSES}, got {len(np.unique(y))}. "
    f"Unique labels in y: {np.unique(y).tolist()}"
)

Shape: (976, 100, 214) (976,)
Class distribution: Counter({0: 170, 4: 170, 3: 166, 1: 160, 2: 156, 5: 154})


In [3]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)
X_val   = torch.tensor(X_val,   dtype=torch.float32)
y_val   = torch.tensor(y_val,   dtype=torch.long)

In [4]:
def mixup_batch(xb, yb, alpha=0.4):
    """
    Mixup augmentation — blends pairs of training samples.
    alpha=0.4 produces stronger blending than 0.2, which helps
    the model interpolate between visually similar shots.
    Returns mixed inputs and soft label tensors.
    """
    lam    = np.random.beta(alpha, alpha)
    idx    = torch.randperm(xb.size(0))
    mixed  = lam * xb + (1 - lam) * xb[idx]
    y_a, y_b = yb, yb[idx]
    return mixed, y_a, y_b, lam


def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)


train_loader = DataLoader(
    TensorDataset(X_train, y_train),
    batch_size=32,
    shuffle=True,
    drop_last=True,
)

val_loader = DataLoader(
    TensorDataset(X_val, y_val),
    batch_size=32,
)

In [5]:
class LSTMModel(nn.Module):
    """
    BiLSTM + temporal attention for cricket shot classification.

    Attention pooling over all T timesteps replaces the bare last-hidden-state
    approach — different shots have discriminative cues at different phases
    (wind-up, contact, follow-through) and attention learns which frames matter.

    hidden_size bumped 256 → 384 to match the larger capacity needed for
    distinguishing the remaining 6 closely-related shots.
    """
    def __init__(
        self,
        input_size:  int,
        hidden_size: int   = 384,
        num_layers:  int   = 3,
        num_classes: int   = 6,
        dropout:     float = 0.4,
    ):
        super().__init__()
        self.input_norm = nn.LayerNorm(input_size)
        self.lstm = nn.LSTM(
            input_size,
            hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout,
            bidirectional=True,
        )
        self.attn    = nn.Linear(hidden_size * 2, 1)
        self.norm    = nn.LayerNorm(hidden_size * 2)
        self.dropout = nn.Dropout(dropout)
        self.fc      = nn.Linear(hidden_size * 2, num_classes)

    def forward(self, x):
        x = self.input_norm(x)
        outputs, _ = self.lstm(x)                       # (B, T, H*2)
        w   = torch.softmax(self.attn(outputs), dim=1)  # (B, T, 1)
        out = (outputs * w).sum(dim=1)                  # (B, H*2)
        out = self.norm(out)
        out = self.dropout(out)
        return self.fc(out)

In [6]:
# ── Class weights — compensate for unequal class sizes ─────────────────────
class_counts = Counter(y_train.tolist())
total        = sum(class_counts.values())

weights = torch.tensor(
    [total / (NUM_CLASSES * class_counts.get(i, 1)) for i in range(NUM_CLASSES)],
    dtype=torch.float32,
)
print('Class weights:', weights)

input_size = X.shape[2]   # 214
NUM_EPOCHS = 200

model     = LSTMModel(input_size=input_size, num_classes=NUM_CLASSES)

criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.1)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=NUM_EPOCHS, eta_min=1e-5
)

print(f'Model parameters: {sum(p.numel() for p in model.parameters()):,}')

Class weights: tensor([0.9559, 1.0156, 1.0400, 0.9848, 0.9559, 1.0569])
Model parameters: 8,940,723


In [7]:
def accuracy(preds, labels):
    return (torch.argmax(preds, dim=1) == labels).float().mean().item()


best_val_acc = 0.0
patience     = 40
counter      = 0
history      = []
USE_MIXUP    = True


for epoch in range(NUM_EPOCHS):
    # ── Train ──────────────────────────────────────────────────────────────
    model.train()
    train_loss, train_acc = 0.0, 0.0

    for xb, yb in train_loader:
        optimizer.zero_grad()

        if USE_MIXUP and np.random.rand() < 0.5:
            xb_mix, ya, yb_, lam = mixup_batch(xb, yb)
            outputs = model(xb_mix)
            loss    = mixup_criterion(criterion, outputs, ya, yb_, lam)
        else:
            outputs = model(xb)
            loss    = criterion(outputs, yb)

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        train_loss += loss.item()
        train_acc  += accuracy(outputs, yb)

    train_loss /= len(train_loader)
    train_acc  /= len(train_loader)

    # ── Validate ───────────────────────────────────────────────────────────
    model.eval()
    val_loss, val_acc = 0.0, 0.0

    with torch.no_grad():
        for xb, yb in val_loader:
            outputs   = model(xb)
            val_loss += criterion(outputs, yb).item()
            val_acc  += accuracy(outputs, yb)

    val_loss /= len(val_loader)
    val_acc  /= len(val_loader)

    scheduler.step()
    lr = scheduler.get_last_lr()[0]

    history.append(dict(epoch=epoch, train_loss=train_loss, train_acc=train_acc,
                        val_loss=val_loss, val_acc=val_acc, lr=lr))

    print(f'Epoch {epoch:3d} | '
          f'Train Loss: {train_loss:.4f}  Acc: {train_acc:.4f} | '
          f'Val Loss: {val_loss:.4f}  Acc: {val_acc:.4f} | '
          f'LR: {lr:.2e}')

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_model.pth')
        counter = 0
    else:
        counter += 1
        if counter >= patience:
            print(f'Early stopping at epoch {epoch}')
            break

print(f'\nTraining complete. Best Val Acc: {best_val_acc:.4f}')

Epoch   0 | Train Loss: 2.0554  Acc: 0.1888 | Val Loss: 1.7288  Acc: 0.2723 | LR: 3.00e-04
Epoch   1 | Train Loss: 1.7729  Acc: 0.2487 | Val Loss: 1.7258  Acc: 0.3170 | LR: 3.00e-04
Epoch   2 | Train Loss: 1.7426  Acc: 0.2930 | Val Loss: 1.7366  Acc: 0.3750 | LR: 3.00e-04
Epoch   3 | Train Loss: 1.7324  Acc: 0.2786 | Val Loss: 1.6645  Acc: 0.2768 | LR: 3.00e-04
Epoch   4 | Train Loss: 1.7086  Acc: 0.2826 | Val Loss: 1.6388  Acc: 0.3259 | LR: 3.00e-04
Epoch   5 | Train Loss: 1.6947  Acc: 0.2943 | Val Loss: 1.6183  Acc: 0.3214 | LR: 2.99e-04
Epoch   6 | Train Loss: 1.6370  Acc: 0.3229 | Val Loss: 1.5924  Acc: 0.3750 | LR: 2.99e-04
Epoch   7 | Train Loss: 1.6196  Acc: 0.3255 | Val Loss: 1.5406  Acc: 0.4152 | LR: 2.99e-04
Epoch   8 | Train Loss: 1.5977  Acc: 0.3438 | Val Loss: 1.5132  Acc: 0.4955 | LR: 2.99e-04
Epoch   9 | Train Loss: 1.5876  Acc: 0.3854 | Val Loss: 1.5237  Acc: 0.4821 | LR: 2.98e-04
Epoch  10 | Train Loss: 1.5234  Acc: 0.3372 | Val Loss: 1.4341  Acc: 0.5223 | LR: 2.98e-04

In [8]:
# Save final model + history
torch.save(model.state_dict(), 'shot_model.pth')

with open('training_history.json', 'w') as f:
    json.dump(history, f, indent=2)

# Per-class report using best checkpoint
model.load_state_dict(torch.load('best_model.pth'))
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for xb, yb in val_loader:
        all_preds.extend(torch.argmax(model(xb), dim=1).tolist())
        all_labels.extend(yb.tolist())

print(classification_report(all_labels, all_preds, target_names=LABEL_NAMES))

                 precision    recall  f1-score   support

    Cover Drive       0.82      0.79      0.81        34
            Cut       0.86      0.78      0.82        32
          Flick       0.90      0.84      0.87        31
Forward Defence       0.81      0.85      0.83        34
           Pull       0.97      0.82      0.89        34
          Sweep       0.70      0.90      0.79        31

       accuracy                           0.83       196
      macro avg       0.84      0.83      0.83       196
   weighted avg       0.84      0.83      0.83       196

